# Projet Final — Fairness in AI
## Classification d'images Chest X-Ray (NIH)

**Equipe :**
- Reina AL MASRI
- Fares SHRETAH
- Mahmoud EL KASSABY

**Cours :** Fairness in AI — Paris-Saclay University

---

### Objectif

Ce notebook constitue le rendu unique du projet final. Le classifieur d'images (ResNet18) est
**fixe et fourni** par le cours via `train_classifieur.py`.
Les images sont les seules entrees du modele ; les metadonnees ne servent qu'a **mesurer et attenuer les biais**.

Le travail se decompose en :
1. Analyse rapide des donnees (desequilibres et biais),
2. Pre-processing par ponderation des instances,
3. Post-processing sur les logits exportes,
4. Comparaison et combinaison des approches,
5. Conclusion argumentee.

> *Le but n'est pas seulement d'obtenir une bonne accuracy,
> mais de documenter les biais, montrer comment le modele les amplifie ou non,
> puis comparer plusieurs strategies de mitigation.*

## Fil conducteur et lien avec les TD

| TD | Apport dans ce projet |
|---|---|
| **TD1 / TD2** | Analyse descriptive, analyse bivariee, fairness metrics sur les labels et predictions |
| **TD3** | Audit du modele : erreurs, sous-groupes, inspection d'images mal classees |
| **TD4** | Pre-processing (ponderation) et post-processing (seuils, reject option) |
| **TD5** | Selection du meilleur compromis sur validation, comparaison des combinaisons |

**Contrainte importante :** les methodes d'in-processing (TD5) ne sont pas compatibles ici sans
modifier le classifieur impose. On conserve l'esprit du TD5 en comparant les combinaisons
`pre-processing + post-processing` sur le split de validation.

In [ ]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display
from PIL import Image
from scipy.stats import chi2_contingency
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

from aif360.sklearn.metrics import (
    average_odds_difference,
    base_rate,
    conditional_demographic_disparity,
    df_bias_amplification,
    disparate_impact_ratio,
    equal_opportunity_difference,
    smoothed_edf,
    statistical_parity_difference,
)

try:
    from aif360.datasets import StandardDataset
    from aif360.algorithms.preprocessing import Reweighing
    AIF360_PREPROCESSING_AVAILABLE = True
except Exception as exc:
    AIF360_PREPROCESSING_AVAILABLE = False
    AIF360_IMPORT_ERROR = exc

from train_classifieur import train_classifier, pred_classifier

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', append=True, category=UserWarning)

In [ ]:
# ---------- Configuration ----------
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

NOTEBOOK_DIR = Path('.')
DATA_DIR = NOTEBOOK_DIR / 'Al_Masri_Reina'
CSV_PATH = DATA_DIR / 'metadata.csv'
EXPERIMENT_CSV = NOTEBOOK_DIR / 'metadata_experiments.csv'
LOG_ROOT = NOTEBOOK_DIR / 'expe_log'

# --- Flags d'execution ---
# Mettre RUN_TRAINING = True pour lancer les entrainements (~20-60 min chacun)
# Mettre FORCE_RETRAIN = True pour re-entrainer meme si un checkpoint existe deja
RUN_TRAINING = False
FORCE_RETRAIN = False
MAX_EPOCHS = 25

VALID_SPLIT_NAME = 'valid'
POSITIVE_LABEL = 'malade'
PRIVILEGED_GENDER = 1   # M = 1 (privilegie), F = 0
PRIVILEGED_AGE = 1       # >60 = 1 (privilegie), <=60 = 0
RANDOM_STATE = 42

LOG_ROOT.mkdir(exist_ok=True)

print('DATA_DIR:', DATA_DIR)
print('CSV_PATH:', CSV_PATH)
print('EXPERIMENT_CSV:', EXPERIMENT_CSV)
print('LOG_ROOT:', LOG_ROOT)
print('RUN_TRAINING:', RUN_TRAINING)
print('MAX_EPOCHS:', MAX_EPOCHS)
print('AIF360 preprocessing available:', AIF360_PREPROCESSING_AVAILABLE)
if not AIF360_PREPROCESSING_AVAILABLE:
    print('AIF360 import issue:', AIF360_IMPORT_ERROR)

In [ ]:
# ===================== Fonctions utilitaires =====================

def get_metrics(
    y_true,
    y_pred=None,
    prot_attr=None,
    priv_group=1,
    pos_label=1,
    sample_weight=None,
):
    """Calcule les metriques de fairness a partir d'aif360 (inspire de TD1/TD2)."""
    metrics = {}
    metrics['base_rate_truth'] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight,
    )
    metrics['statistical_parity_difference'] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
        priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight,
    )
    metrics['disparate_impact_ratio'] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
        priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight,
    )
    if y_pred is not None:
        metrics['base_rate_preds'] = base_rate(
            y_true=y_pred, pos_label=pos_label, sample_weight=sample_weight,
        )
        metrics['equal_opportunity_difference'] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight,
        )
        metrics['average_odds_difference'] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            priv_group=priv_group, pos_label=pos_label, sample_weight=sample_weight,
        )
        if len(set(y_pred)) > 1:
            metrics['conditional_demographic_disparity'] = conditional_demographic_disparity(
                y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
                pos_label=pos_label, sample_weight=sample_weight,
            )
        else:
            metrics['conditional_demographic_disparity'] = None
        metrics['smoothed_edf'] = smoothed_edf(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            pos_label=pos_label, sample_weight=sample_weight,
        )
        metrics['df_bias_amplification'] = df_bias_amplification(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            pos_label=pos_label, sample_weight=sample_weight,
        )
        metrics['balanced_accuracy_score'] = balanced_accuracy_score(
            y_true=y_true, y_pred=y_pred, sample_weight=sample_weight,
        )
    return metrics


def clean_age_series(age_series, max_valid_age=100):
    age = pd.to_numeric(age_series, errors='coerce')
    return age.where(age.between(0, max_valid_age))


def make_age_group(age_series):
    return pd.cut(
        age_series, bins=[0, 40, 60, 100],
        labels=['<=40', '41-60', '>60'], include_lowest=True,
    )


def normalize_weights(values):
    values = pd.Series(values, dtype=float)
    return values / values.mean()


def safe_mode(series):
    mode = series.mode(dropna=True)
    return mode.iloc[0] if not mode.empty else np.nan


def add_common_columns(df):
    out = df.copy()
    out['Patient Age Clean'] = clean_age_series(out['Patient Age'])
    out['age_group'] = make_age_group(out['Patient Age Clean']).astype('object').fillna('Unknown')
    out['gender_binary'] = (out['Patient Gender'] == 'M').astype(int)
    out['age_binary'] = out['Patient Age Clean'].gt(60).fillna(False).astype(int)
    if 'label' in out.columns:
        out['has_finding'] = (out['label'] == POSITIVE_LABEL).astype(int)
    elif 'labels' in out.columns:
        out['has_finding'] = (out['labels'] == POSITIVE_LABEL).astype(int)
    else:
        out['has_finding'] = (out['Finding Labels'] != 'No Finding').astype(int)
    return out


def build_patient_table(df_img):
    df_patient = (
        df_img.groupby('Patient ID')
        .agg({
            'Patient Age Clean': 'first',
            'Patient Gender': 'first',
            'View Position': safe_mode,
            'has_finding': 'max',
            'Image Index': 'count',
        })
        .rename(columns={'Image Index': 'num_visits'})
        .reset_index()
    )
    df_patient['age_group'] = make_age_group(df_patient['Patient Age Clean']).astype('object').fillna('Unknown')
    df_patient['gender_binary'] = (df_patient['Patient Gender'] == 'M').astype(int)
    df_patient['age_binary'] = df_patient['Patient Age Clean'].gt(60).fillna(False).astype(int)
    return df_patient


def analyse_univariee(variable, df):
    counts = df[variable].value_counts(dropna=False)
    freqs = df[variable].value_counts(dropna=False, normalize=True).mul(100)
    mode = df[variable].mode(dropna=True)
    mode_value = mode.iloc[0] if not mode.empty else np.nan
    summary = pd.DataFrame({'effectif': counts, 'frequence_pct': freqs.round(2)})
    return summary, mode_value


def rapport_correlation(df, qualitative, quantitative):
    tmp = df[[qualitative, quantitative]].dropna()
    if tmp.empty:
        return np.nan
    groups = tmp.groupby(qualitative)[quantitative]
    mean_total = tmp[quantitative].mean()
    n_total = len(tmp)
    var_inter = sum(len(g) * (g.mean() - mean_total) ** 2 for _, g in groups) / n_total
    var_total = tmp[quantitative].var(ddof=0)
    return 0.0 if var_total == 0 else var_inter / var_total


def analyse_bivariee_quali(df, left, right):
    contingency = pd.crosstab(df[left], df[right])
    profiles = pd.crosstab(df[left], df[right], normalize='index').mul(100)
    chi2, p_value, dof, _ = chi2_contingency(contingency)
    return contingency, profiles.round(2), chi2, p_value, dof


def infer_class_names(datadir):
    train_dir = Path(datadir) / 'train'
    return sorted([
        name for name in os.listdir(train_dir)
        if (train_dir / name).is_dir() and not name.startswith('.')
    ])

In [ ]:
CLASS_NAMES = infer_class_names(DATA_DIR)
POSITIVE_INDEX = CLASS_NAMES.index(POSITIVE_LABEL)
NEGATIVE_LABEL = [label for label in CLASS_NAMES if label != POSITIVE_LABEL][0]

print('Classes detectees:', CLASS_NAMES)
print('Label positif:', POSITIVE_LABEL)
print('Label negatif:', NEGATIVE_LABEL)

---
## 1. Chargement et exploration du jeu de donnees

Chaque membre de l'equipe a recu un sous-ensemble du dataset NIH Chest X-Ray.
La structure fournie est :
```
Al_Masri_Reina/
  train/
    malade/
    sain/
  valid/
    malade/
    sain/
  metadata.csv
```

Le fichier `metadata.csv` contient les metadonnees par image : age, genre, position de vue, pathologies.
Le modele ResNet18 ne voit **que les images**, mais ces metadonnees sont essentielles pour analyser les biais.

In [ ]:
df_img_raw = pd.read_csv(CSV_PATH)
df_img = add_common_columns(df_img_raw)
df_patient = build_patient_table(df_img)

print('Forme au niveau image   :', df_img.shape)
print('Forme au niveau patient :', df_patient.shape)
print()
print('Colonnes du CSV :')
display(pd.Series(df_img.columns.tolist()))
display(df_img.head(3))

In [ ]:
# Resume statistique rapide du dataset
current_facts = {
    'Nombre d\'images': len(df_img),
    'Nombre de patients': df_patient['Patient ID'].nunique(),
    'Images train': int((df_img['train_valid'] == 'train').sum()),
    'Images valid': int((df_img['train_valid'] == 'valid').sum()),
    'Malades': int((df_img['label'] == 'malade').sum()),
    'Sains': int((df_img['label'] == 'sain').sum()),
    'Hommes (M)': int((df_img['Patient Gender'] == 'M').sum()),
    'Femmes (F)': int((df_img['Patient Gender'] == 'F').sum()),
    'Vue PA': int((df_img['View Position'] == 'PA').sum()),
    'Vue AP': int((df_img['View Position'] == 'AP').sum()),
    'Outliers age > 100': int((pd.to_numeric(df_img['Patient Age'], errors='coerce') > 100).sum()),
}

facts_df = pd.DataFrame.from_dict(current_facts, orient='index', columns=['valeur'])
display(facts_df)

**Constats sur le dataset :**

- Le dataset contient environ 5 000 images reparties entre `train` et `valid`.
- Il n'y a **pas de split test separe** : le `valid` sert a la fois pour l'evaluation et le choix des seuils.
  Cette limitation doit etre assumee dans nos conclusions.
- Le ratio malade/sain montre un leger desequilibre qu'il faudra quantifier.
- Le ratio hommes/femmes revele une sur-representation masculine, classique dans les datasets medicaux.
- Les outliers d'age (> 100 ans) sont rares et seront filtres automatiquement par `clean_age_series`.

---
## 2. Visualisation d'images du dataset

Le modele ResNet18 ne consomme que les radiographies, pas les metadonnees.
Il est donc essentiel de montrer a quoi ressemblent ces images, et de rappeler que
des attributs comme `View Position` (AP vs PA) sont **directement lisibles dans l'image**.

Comme le montrent les slides du cours, le ResNet18 est capable de retrouver le genre,
l'age et la position de vue a partir de l'image seule. C'est pour cette raison que
les metadonnees sont necessaires pour analyser et corriger les biais, meme si le modele
ne les recoit jamais explicitement.

In [ ]:
def get_true_label_col(df):
    if 'label' in df.columns:
        return 'label'
    if 'labels' in df.columns:
        return 'labels'
    raise KeyError('Aucune colonne de label verite disponible.')


def resolve_image_path(row):
    split = row['train_valid']
    if 'label' in row.index:
        label = row['label']
    elif 'labels' in row.index:
        label = row['labels']
    else:
        raise KeyError('Label introuvable pour resoudre le chemin image.')
    return DATA_DIR / split / label / row['Image Index']


def show_image_grid(df, title, rows=2, cols=4, random_state=RANDOM_STATE):
    sample_n = min(len(df), rows * cols)
    if sample_n == 0:
        print('Aucune image disponible pour', title)
        return
    sample = df.sample(sample_n, random_state=random_state)
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
        img_path = resolve_image_path(row)
        img = Image.open(img_path).convert('L')
        ax.imshow(img, cmap='gray')
        cell_title = (
            f"{row['Image Index']}\n"
            f"{row['train_valid']} | {row[get_true_label_col(pd.DataFrame([row]))]} | "
            f"{row['Patient Gender']} | {row['View Position']}"
        )
        ax.set_title(cell_title, fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
show_image_grid(
    df_img[(df_img['train_valid'] == 'train') & (df_img['label'] == 'malade')],
    title="Exemples d'images train — classe malade",
)

show_image_grid(
    df_img[(df_img['train_valid'] == 'train') & (df_img['View Position'] == 'AP')],
    title="Exemples d'images train — View Position AP",
)

In [ ]:
def stratified_image_examples(df, row_group, col_group, split='train',
                              samples_per_cell=1, random_state=RANDOM_STATE):
    """Grille d'images stratifiee par deux variables categorielles."""
    subset = df[df['train_valid'] == split].copy()
    row_values = sorted([x for x in subset[row_group].dropna().unique()])
    col_values = sorted([x for x in subset[col_group].dropna().unique()])
    nrows = len(row_values)
    ncols = len(col_values) * samples_per_cell
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.5 * nrows))
    axes = np.atleast_1d(axes).reshape(nrows, ncols)
    for ax in axes.ravel():
        ax.axis('off')
    for i, row_value in enumerate(row_values):
        for j, col_value in enumerate(col_values):
            cell_df = subset[(subset[row_group] == row_value) & (subset[col_group] == col_value)]
            if cell_df.empty:
                continue
            chosen = cell_df.sample(min(samples_per_cell, len(cell_df)), random_state=random_state)
            for k, (_, row) in enumerate(chosen.iterrows()):
                ax = axes[i, j * samples_per_cell + k]
                img = Image.open(resolve_image_path(row)).convert('L')
                ax.imshow(img, cmap='gray')
                ax.set_title(f'{row_group}={row_value}\n{col_group}={col_value}', fontsize=9)
    plt.tight_layout()
    plt.show()

stratified_image_examples(df_img, row_group='Patient Gender', col_group='label')
stratified_image_examples(df_img, row_group='View Position', col_group='label')

---
## 3. Analyse rapide des donnees : desequilibres et biais

*Conformement a la consigne, cette analyse est plus courte que le mi-projet mais doit montrer :*
- *les desequilibres de distribution (analyse univariee),*
- *les liens entre attributs sensibles et cible (analyse bivariee),*
- *des indices de biais ou de proxy susceptibles d'etre appris par le modele.*

### 3.1 Analyse univariee

L'analyse univariee revele les **desequilibres** dans les effectifs.
Un desequilibre n'est pas un biais en soi, mais il en est souvent le premier indicateur.

In [ ]:
for variable in ['Patient Gender', 'View Position', 'age_group']:
    summary, mode_value = analyse_univariee(variable, df_patient)
    print(f'--- Analyse univariee de {variable} ---')
    display(summary)
    print('Mode:', mode_value)
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(data=df_img, x='label', order=['sain', 'malade'], ax=axes[0], color='#6fb98f')
axes[0].set_title('Distribution des labels (niveau image)')

sns.countplot(data=df_img, x='Patient Gender', order=['F', 'M'], ax=axes[1], color='#f2b880')
axes[1].set_title('Distribution du genre (niveau image)')

age_plot_df = df_patient.dropna(subset=['Patient Age Clean'])
sns.histplot(age_plot_df['Patient Age Clean'], bins=25, kde=True, ax=axes[2], color='#1f6f8b')
axes[2].set_title("Distribution de l'age (niveau patient)")

plt.tight_layout()
plt.show()

### 3.2 Analyse bivariee : attribut sensible x cible

L'analyse bivariee permet de detecter des **biais** : une difference de taux de pathologie
entre les groupes sensibles. On utilise :
- le **rapport de correlation eta²** pour le lien quali-quanti (Genre x Age),
- le **test du Chi²** pour les croisements quali-quali (Genre x has_finding, etc.),
- des **profils lignes** pour visualiser les differences de taux.

> **Rappel (TD2)** : Un biais peut avoir une explication (ex. Paradoxe de Simpson),
> mais il reste factuel. Un modele ML a tendance a **amplifier** les biais presents.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

gender_ctab = pd.crosstab(df_img['Patient Gender'], df_img['label'], normalize='index').mul(100)
gender_ctab.plot(kind='bar', ax=axes[0], title='Taux de label par genre')
axes[0].set_ylabel('Pourcentage')
axes[0].legend(title='label')

age_ctab = pd.crosstab(df_patient['age_group'], df_patient['has_finding'], normalize='index').mul(100)
age_ctab.rename(columns={0: 'sain', 1: 'malade'}).plot(
    kind='bar', ax=axes[1], title='Taux de pathologie par groupe d\'age')
axes[1].set_ylabel('Pourcentage')
axes[1].legend(title='has_finding')

plt.tight_layout()
plt.show()

display(gender_ctab.round(2))
display(age_ctab.round(2))

In [ ]:
# Rapports de correlation et tests du Chi²
eta_gender_age = rapport_correlation(df_patient, 'Patient Gender', 'Patient Age Clean')
eta_gender_visits = rapport_correlation(df_patient, 'Patient Gender', 'num_visits')

comparisons = [
    ('Patient Gender', 'has_finding'),
    ('Patient Gender', 'View Position'),
    ('Patient Gender', 'age_group'),
    ('View Position', 'has_finding'),
    ('age_group', 'has_finding'),
]

print(f'eta²(Genre, Age)        = {eta_gender_age:.4f}')
print(f'eta²(Genre, num_visits) = {eta_gender_visits:.4f}')
print()

for left, right in comparisons:
    contingency, profiles, chi2, p_value, dof = analyse_bivariee_quali(df_patient, left, right)
    print(f'--- {left} x {right} ---')
    display(contingency)
    display(profiles)
    sig = 'OUI (p < 0.05)' if p_value < 0.05 else 'NON (p >= 0.05)'
    print(f'Chi2={chi2:.4f}, p-value={p_value:.6g}, ddl={dof} --> Significatif ? {sig}')
    print()

In [ ]:
# Prevalence des pathologies les plus frequentes par genre
all_labels = df_img['Finding Labels'].str.split('|').explode()
top_diseases = all_labels[all_labels != 'No Finding'].value_counts().head(8).index.tolist()

prevalence_rows = []
for disease in top_diseases:
    patient_flag = (
        df_img.assign(flag=df_img['Finding Labels'].str.contains(disease, regex=False))
        .groupby('Patient ID')
        .agg({'flag': 'max', 'Patient Gender': 'first'})
        .reset_index()
    )
    for gender in ['F', 'M']:
        rate = patient_flag.loc[patient_flag['Patient Gender'] == gender, 'flag'].mean() * 100
        prevalence_rows.append({'Pathologie': disease, 'Genre': gender, 'Prevalence (%)': rate})

df_prev = pd.DataFrame(prevalence_rows)
fig = px.bar(
    df_prev, x='Pathologie', y='Prevalence (%)', color='Genre', barmode='group',
    title='Prevalence des pathologies les plus frequentes par genre',
)
fig.update_layout(xaxis_tickangle=-40)
fig.show()

### 3.3 Interpretation de l'analyse descriptive

**Desequilibres observes :**
- **Genre** : le dataset est desequilibre en faveur des hommes.
  Ce desequilibre est classique dans les datasets medicaux et constitue un premier facteur de risque.
- **Age** : la distribution est concentree entre 40 et 70 ans, avec peu de patients jeunes.
  Les plus de 60 ans ont un taux de pathologie significativement plus eleve, ce qui est medicalement attendu.
- **View Position** : la repartition AP / PA n'est pas equilibree.
  C'est important car le ResNet18 peut inferer cette information directement depuis l'image.

**Biais detectes :**
- Le croisement **Genre x has_finding** revele une difference de taux de pathologie entre hommes et femmes.
  Si le test du Chi² est significatif (p < 0.05), ce biais est statistiquement reel.
- Le croisement **View Position x has_finding** peut egalement montrer un biais.
  La position AP est souvent associee a des patients alites, donc plus malades.
- La prevalence des pathologies specifiques differe entre genres, ce qui confirme le biais global.

> **Lien avec le modele :** Le ResNet18 peut retrouver le genre, l'age et la position de vue
> a partir de l'image seule (cf. slides du cours). Si un biais existe dans les labels,
> le modele risque de l'amplifier via ces proxies visuels.

---
## 4. Fairness des labels et analyse proxy (avant tout modele)

Avant d'entrainer le modele, on mesure le **biais intrinseque** present dans les labels
du dataset. En passant `y_pred = y_true`, on obtient les metriques de fairness sur les donnees
elles-memes (cf. TD1/TD2).

**Metriques utilisees :**
| Metrique | Signification | Ideal |
|---|---|---|
| **SPD** (Statistical Parity Difference) | Difference de taux de positifs entre groupes | 0 |
| **DIR** (Disparate Impact Ratio) | Ratio des taux de positifs (regle des 80%) | 1 |
| **EOD** (Equal Opportunity Difference) | Difference de TPR entre groupes | 0 |
| **AOD** (Average Odds Difference) | Moyenne des ecarts FPR et TPR | 0 |

In [ ]:
label_gender_metrics = get_metrics(
    y_true=df_patient['has_finding'].to_numpy(),
    y_pred=df_patient['has_finding'].to_numpy(),
    prot_attr=df_patient['gender_binary'].to_numpy(),
    priv_group=PRIVILEGED_GENDER,
    pos_label=1,
)

label_age_metrics = get_metrics(
    y_true=df_patient['has_finding'].to_numpy(),
    y_pred=df_patient['has_finding'].to_numpy(),
    prot_attr=df_patient['age_binary'].to_numpy(),
    priv_group=PRIVILEGED_AGE,
    pos_label=1,
)

label_fairness_df = pd.DataFrame({
    'genre (M=priv)': label_gender_metrics,
    'age (>60=priv)': label_age_metrics,
}).round(4)

print('Metriques de fairness sur les LABELS (y_pred = y_true) :')
display(label_fairness_df)

In [ ]:
# Analyse proxy : quelles variables sont correlees avec le genre ?
proxy_summary = pd.DataFrame([
    {'lien': 'Genre x Age', 'mesure': 'eta²',
     'valeur': rapport_correlation(df_patient, 'Patient Gender', 'Patient Age Clean')},
    {'lien': 'Genre x num_visits', 'mesure': 'eta²',
     'valeur': rapport_correlation(df_patient, 'Patient Gender', 'num_visits')},
    {'lien': 'Genre x View Position', 'mesure': 'chi2 p-value',
     'valeur': analyse_bivariee_quali(df_patient, 'Patient Gender', 'View Position')[3]},
    {'lien': 'Genre x age_group', 'mesure': 'chi2 p-value',
     'valeur': analyse_bivariee_quali(df_patient, 'Patient Gender', 'age_group')[3]},
    {'lien': 'View Position x target', 'mesure': 'chi2 p-value',
     'valeur': analyse_bivariee_quali(df_patient, 'View Position', 'has_finding')[3]},
])

print('Resume des liens proxy :')
display(proxy_summary)
print()
print('Interpretation : si une variable est fortement correlee avec le genre')
print('ET avec la cible, elle peut servir de proxy et transmettre le biais au modele,')
print('meme sans acces direct a l\'attribut sensible.')

---
## 5. Pre-processing : strategies de ponderation

Le classifieur impose utilise un `WeightedRandomSampler` qui s'appuie sur une colonne de poids
dans le CSV. C'est le **seul levier de pre-processing** compatible avec le pipeline.

On teste plusieurs strategies de ponderation (inspirees de TD4) :

| Strategie | Principe | Colonne |
|---|---|---|
| **Baseline** | Poids uniformes (1.0) | `WEIGHTS_BASELINE` |
| **Course** | Poids fournis par le cours | `WEIGHTS_COURSE` |
| **Class balanced** | Frequence inverse par classe | `WEIGHTS_CLASS_BALANCED` |
| **Gender Reweighing** | Reweighing AIF360 sur le genre | `WEIGHTS_GENDER_RW` |
| **Age Reweighing** | Reweighing AIF360 sur l'age | `WEIGHTS_AGE_RW` |
| **Gender x Label** | Frequence inverse par genre x label | `WEIGHTS_GENDER_LABEL` |
| **View x Label** | Frequence inverse par view position x label | `WEIGHTS_VIEW_LABEL` |

> **Pourquoi pas DisparateImpactRemover ?** Cette methode modifie les features tabulaires.
> Or ici le modele ne consomme que l'image : modifier les metadonnees n'aurait aucun effet.

In [ ]:
def inverse_frequency_weights(df, group_cols):
    """Poids = frequence inverse du groupe. Plus un groupe est petit, plus il pese."""
    counts = df.groupby(group_cols)['Image Index'].transform('count')
    return normalize_weights(1.0 / counts)


def build_reweighing_weights(df, protected_col, privileged_value=1):
    """Reweighing AIF360 : ajuste les poids pour egaliser les taux de base entre groupes."""
    if not AIF360_PREPROCESSING_AVAILABLE:
        print(f'AIF360 non disponible, poids uniformes pour {protected_col}')
        return normalize_weights(np.ones(len(df)))

    rw_df = df[[protected_col, 'has_finding']].copy()
    rw_df['dummy_feature'] = 0

    dataset = StandardDataset(
        df=rw_df,
        label_name='has_finding',
        favorable_classes=[1],
        protected_attribute_names=[protected_col],
        privileged_classes=[[privileged_value]],
        categorical_features=[],
        features_to_keep=['dummy_feature', protected_col, 'has_finding'],
    )

    rw = Reweighing(
        unprivileged_groups=[{protected_col: 0}],
        privileged_groups=[{protected_col: privileged_value}],
    )
    dataset_rw = rw.fit_transform(dataset)
    return normalize_weights(dataset_rw.instance_weights)


# Calcul de toutes les strategies de ponderation
df_model = df_img.copy()
df_model['WEIGHTS_BASELINE'] = 1.0
df_model['WEIGHTS_COURSE'] = normalize_weights(
    pd.to_numeric(df_model['WEIGHTS'], errors='coerce').fillna(1.0)
)
df_model['WEIGHTS_CLASS_BALANCED'] = inverse_frequency_weights(df_model, ['label'])
df_model['WEIGHTS_GENDER_RW'] = build_reweighing_weights(
    df_model, 'gender_binary', privileged_value=PRIVILEGED_GENDER
)
df_model['WEIGHTS_AGE_RW'] = build_reweighing_weights(
    df_model, 'age_binary', privileged_value=PRIVILEGED_AGE
)
df_model['WEIGHTS_GENDER_LABEL'] = inverse_frequency_weights(df_model, ['Patient Gender', 'label'])
df_model['WEIGHTS_VIEW_LABEL'] = inverse_frequency_weights(df_model, ['View Position', 'label'])

# Sauvegarder le CSV enrichi
df_model.to_csv(EXPERIMENT_CSV, index=False)

weight_columns = [
    'WEIGHTS_BASELINE', 'WEIGHTS_COURSE', 'WEIGHTS_CLASS_BALANCED',
    'WEIGHTS_GENDER_RW', 'WEIGHTS_AGE_RW', 'WEIGHTS_GENDER_LABEL', 'WEIGHTS_VIEW_LABEL',
]

print('CSV enrichi sauvegarde dans :', EXPERIMENT_CSV)
display(df_model[['Image Index', 'label', 'Patient Gender', 'View Position', 'age_group'] + weight_columns].head())

In [ ]:
# Diagnostic des poids : statistiques descriptives
weight_diagnostics = []
for col in weight_columns:
    stats = df_model[col].describe()[['mean', 'std', 'min', 'max']].to_dict()
    stats['strategie'] = col.replace('WEIGHTS_', '')
    weight_diagnostics.append(stats)

weight_diagnostics_df = pd.DataFrame(weight_diagnostics).set_index('strategie').round(4)
display(weight_diagnostics_df)

# Verifier que les poids sont normalises (moyenne ~1)
print('\nVerification : la moyenne de chaque strategie doit etre ~1.0')
for col in weight_columns:
    print(f'  {col}: mean = {df_model[col].mean():.4f}')

---
## 6. Entrainement du modele et prediction

`train_classifieur.py` reste **inchange**. Le notebook :
1. Ecrit un CSV enrichi avec les nouvelles colonnes de poids,
2. Lance `train_classifier(...)` pour chaque strategie,
3. Lance `pred_classifier(...)` pour exporter les logits,
4. Compare les sorties sur le split `valid`.

> **Important :** Mettre `RUN_TRAINING = True` dans la cellule de configuration
> pour lancer les entrainements (~20-60 min chacun).
> Une fois les checkpoints et preds.csv generes, remettre `RUN_TRAINING = False`
> pour eviter de relancer par erreur.

In [ ]:
EXPERIMENTS = [
    {'run': 'baseline_unweighted',   'weights_col': 'WEIGHTS_BASELINE',       'active': True},
    {'run': 'course_weights',        'weights_col': 'WEIGHTS_COURSE',         'active': True},
    {'run': 'class_balanced',        'weights_col': 'WEIGHTS_CLASS_BALANCED', 'active': True},
    {'run': 'gender_rw',             'weights_col': 'WEIGHTS_GENDER_RW',      'active': True},
    {'run': 'age_rw',                'weights_col': 'WEIGHTS_AGE_RW',         'active': True},
    {'run': 'gender_label_balanced', 'weights_col': 'WEIGHTS_GENDER_LABEL',   'active': True},
    {'run': 'view_label_balanced',   'weights_col': 'WEIGHTS_VIEW_LABEL',     'active': True},
]

display(pd.DataFrame(EXPERIMENTS))

In [ ]:
def find_checkpoint(logdir):
    matches = sorted(glob.glob(str(Path(logdir) / '*.ckpt')))
    return matches[0] if matches else None


def run_single_experiment(run_name, weights_col, csv_path=EXPERIMENT_CSV):
    """Lance l'entrainement et/ou la prediction pour une experience."""
    logdir = LOG_ROOT / run_name
    preds_path = logdir / 'preds.csv'
    ckpt_path = find_checkpoint(logdir)

    should_train = RUN_TRAINING and (FORCE_RETRAIN or ckpt_path is None)
    should_predict = RUN_TRAINING and (FORCE_RETRAIN or not preds_path.exists())

    if should_train:
        ckpt_path, ckpt_score = train_classifier(
            logdir=str(logdir),
            datadir=str(DATA_DIR),
            csv=str(csv_path),
            weights_col=weights_col,
            max_epochs=MAX_EPOCHS,
        )
        print(f'{run_name} entraine — best score = {ckpt_score:.4f}')

    if should_predict:
        ckpt_path = ckpt_path or find_checkpoint(logdir)
        if ckpt_path is None:
            raise FileNotFoundError(f'Aucun checkpoint disponible pour {run_name}')
        pred_classifier(
            datadir=str(DATA_DIR),
            ckpt_path=str(ckpt_path),
            csv_in=str(csv_path),
            csv_out=str(preds_path),
        )
        print(f'{run_name} predictions exportees dans {preds_path}')

    return {
        'run': run_name,
        'weights_col': weights_col,
        'logdir': str(logdir),
        'ckpt_path': ckpt_path or find_checkpoint(logdir),
        'preds_path': str(preds_path),
        'preds_exists': preds_path.exists(),
    }


experiment_registry = []
for exp in EXPERIMENTS:
    if exp['active']:
        experiment_registry.append(run_single_experiment(exp['run'], exp['weights_col']))

experiment_registry_df = pd.DataFrame(experiment_registry)
display(experiment_registry_df)

# Afficher les experiences pretes vs manquantes
n_ready = experiment_registry_df['preds_exists'].sum()
n_total = len(experiment_registry_df)
print(f'\n{n_ready}/{n_total} experiences ont des predictions disponibles.')
if n_ready < n_total:
    print('Pour lancer les experiences manquantes, mettre RUN_TRAINING = True et re-executer.')

---
## 7. Audit du modele sur les predictions

Cette section est inspiree des **TD1 et TD3** :
- **Performance globale** (balanced accuracy) par strategie de ponderation,
- **Fairness des predictions** (SPD, DIR, EOD, AOD) par attribut sensible,
- **Analyse par sous-groupes** : taux d'erreur, TPR, FPR par genre, age, view position,
- **Inspection d'images mal classees** : identifier les patterns d'erreur.

> **Point cle :** La balanced accuracy est la metrique de performance principale car elle
> compense le desequilibre de classes. Mais une bonne accuracy ne garantit pas la fairness.

In [ ]:
def softmax_np(logits):
    logits = np.asarray(logits, dtype=float)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def prepare_prediction_frame(df_pred):
    """Enrichit le dataframe de predictions avec les colonnes necessaires a l'audit."""
    df_eval = add_common_columns(df_pred)
    logit_cols = sorted(
        [col for col in df_eval.columns if col.startswith('preds_logit')],
        key=lambda col: int(col.replace('preds_logit', '')),
    )
    probs = softmax_np(df_eval[logit_cols].to_numpy())
    df_eval['score_malade'] = probs[:, POSITIVE_INDEX]
    df_eval['y_true'] = (df_eval['labels'] == POSITIVE_LABEL).astype(int)
    df_eval['y_pred'] = (df_eval['preds'] == POSITIVE_LABEL).astype(int)
    df_eval['error'] = (df_eval['y_true'] != df_eval['y_pred']).astype(int)
    return df_eval


def fairness_summary(y_true, y_pred, prot_attr, priv_group):
    metrics = get_metrics(
        y_true=np.asarray(y_true), y_pred=np.asarray(y_pred),
        prot_attr=np.asarray(prot_attr), priv_group=priv_group, pos_label=1,
    )
    return {
        'SPD': metrics['statistical_parity_difference'],
        'DIR': metrics['disparate_impact_ratio'],
        'EOD': metrics['equal_opportunity_difference'],
        'AOD': metrics['average_odds_difference'],
    }


def evaluate_run(df_pred, run_name, stage='raw', split_name=VALID_SPLIT_NAME,
                 y_pred_override=None):
    """Evalue une experience sur le split de validation."""
    df_eval = prepare_prediction_frame(df_pred)
    df_eval = df_eval[df_eval['train_valid'] == split_name].copy()
    y_true = df_eval['y_true'].to_numpy()
    y_pred = (df_eval['y_pred'].to_numpy() if y_pred_override is None
              else np.asarray(y_pred_override))

    gender_metrics = fairness_summary(y_true, y_pred, df_eval['gender_binary'], PRIVILEGED_GENDER)
    age_metrics = fairness_summary(y_true, y_pred, df_eval['age_binary'], PRIVILEGED_AGE)

    return {
        'run': run_name, 'stage': stage, 'split': split_name, 'n': len(df_eval),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'accuracy': accuracy_score(y_true, y_pred),
        'gender_SPD': gender_metrics['SPD'], 'gender_DIR': gender_metrics['DIR'],
        'gender_EOD': gender_metrics['EOD'], 'gender_AOD': gender_metrics['AOD'],
        'age_SPD': age_metrics['SPD'], 'age_DIR': age_metrics['DIR'],
        'age_EOD': age_metrics['EOD'], 'age_AOD': age_metrics['AOD'],
    }


def subgroup_report(df_eval, group_col):
    """Rapport detaille par sous-groupe (inspire de TD3)."""
    rows = []
    for group_value, group_df in df_eval.groupby(group_col):
        y_true = group_df['y_true'].to_numpy()
        y_pred = group_df['y_pred'].to_numpy()
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        tpr = tp / (tp + fn) if (tp + fn) else np.nan
        tnr = tn / (tn + fp) if (tn + fp) else np.nan
        fpr = fp / (fp + tn) if (fp + tn) else np.nan
        fnr = fn / (fn + tp) if (fn + tp) else np.nan
        rows.append({
            group_col: group_value, 'n': len(group_df),
            'prevalence': group_df['y_true'].mean(),
            'taux_prediction_positive': group_df['y_pred'].mean(),
            'taux_erreur': (group_df['y_true'] != group_df['y_pred']).mean(),
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'TPR': tpr, 'TNR': tnr, 'FPR': fpr, 'FNR': fnr,
        })
    return pd.DataFrame(rows).sort_values(group_col)


def plot_group_confusion_matrices(df_eval, group_col, title_prefix):
    groups = sorted(df_eval[group_col].dropna().unique())
    fig, axes = plt.subplots(1, len(groups), figsize=(5 * len(groups), 4))
    axes = np.atleast_1d(axes)
    for ax, value in zip(axes, groups):
        group_df = df_eval[df_eval[group_col] == value]
        cm = confusion_matrix(group_df['y_true'], group_df['y_pred'], labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                    xticklabels=['sain', 'malade'], yticklabels=['sain', 'malade'])
        ax.set_title(f'{title_prefix} — {group_col}={value}')
        ax.set_xlabel('Prediction')
        ax.set_ylabel('Verite')
    plt.tight_layout()
    plt.show()

In [ ]:
# Charger toutes les predictions disponibles et evaluer
available_predictions = {}
raw_results = []

for exp in EXPERIMENTS:
    if not exp['active']:
        continue
    preds_path = LOG_ROOT / exp['run'] / 'preds.csv'
    if preds_path.exists():
        df_pred = pd.read_csv(preds_path)
        available_predictions[exp['run']] = df_pred
        raw_results.append(evaluate_run(df_pred, exp['run'], stage='raw'))

raw_results_df = pd.DataFrame(raw_results)
if raw_results_df.empty:
    print('Aucun preds.csv disponible.')
    print('Mettez RUN_TRAINING=True ou reutilisez des runs deja presents.')
else:
    raw_results_df = raw_results_df.sort_values('balanced_accuracy', ascending=False).reset_index(drop=True)
    print(f'{len(raw_results_df)} experience(s) evaluee(s) :')
    display(raw_results_df.round(4))

In [ ]:
# Visualisation des performances et de la fairness par strategie de pre-processing
if not raw_results_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Balanced accuracy
    raw_sorted = raw_results_df.sort_values('balanced_accuracy', ascending=True)
    axes[0].barh(raw_sorted['run'], raw_sorted['balanced_accuracy'], color='#1f6f8b')
    axes[0].set_xlabel('Balanced Accuracy')
    axes[0].set_title('Performance par strategie de ponderation')
    for i, (_, row) in enumerate(raw_sorted.iterrows()):
        axes[0].text(row['balanced_accuracy'] + 0.002, i, f"{row['balanced_accuracy']:.4f}", va='center')

    # |SPD genre|
    raw_sorted_spd = raw_results_df.sort_values('gender_SPD', key=lambda x: x.abs(), ascending=True)
    colors = ['#2ecc71' if abs(v) < 0.05 else '#e74c3c' for v in raw_sorted_spd['gender_SPD']]
    axes[1].barh(raw_sorted_spd['run'], raw_sorted_spd['gender_SPD'].abs(), color=colors)
    axes[1].set_xlabel('|SPD genre|')
    axes[1].set_title('Ecart de parite statistique (genre) par strategie')
    axes[1].axvline(x=0.05, color='orange', linestyle='--', label='seuil 0.05')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# Audit detaille : sous-groupes et matrices de confusion
if not raw_results_df.empty:
    # Choisir la meilleure strategie pour l'audit detaille
    baseline_name = 'baseline_unweighted'
    best_raw_name = raw_results_df.iloc[0]['run']
    audit_name = best_raw_name if best_raw_name in available_predictions else baseline_name

    if audit_name in available_predictions:
        audit_df = prepare_prediction_frame(available_predictions[audit_name])
        audit_df = audit_df[audit_df['train_valid'] == VALID_SPLIT_NAME].copy()

        print(f'=== Audit detaille de : {audit_name} ===')
        print()

        print('--- Rapport par genre ---')
        display(subgroup_report(audit_df, 'Patient Gender').round(4))
        print()

        print('--- Rapport par groupe d\'age ---')
        display(subgroup_report(audit_df, 'age_group').round(4))
        print()

        print('--- Rapport par View Position ---')
        display(subgroup_report(audit_df, 'View Position').round(4))

        plot_group_confusion_matrices(audit_df, 'Patient Gender', title_prefix=audit_name)
        plot_group_confusion_matrices(audit_df, 'age_group', title_prefix=audit_name)

### 7.1 Analyse des erreurs et inspection d'images

L'inspection visuelle des images mal classees permet de comprendre les patterns d'erreur du modele.
On regarde en particulier :
- les **faux negatifs** (malades predits sains) : les plus dangereux en contexte medical,
- les **cas proches de la frontiere** (score ~ 0.5) : les plus susceptibles d'etre corriges par post-processing.

In [ ]:
def show_error_gallery(df_eval, title, filter_mask=None, n=8, hardest=False):
    """Affiche une galerie d'images mal classees."""
    subset = df_eval.copy()
    if filter_mask is not None:
        subset = subset.loc[filter_mask].copy()
    subset = subset[subset['error'] == 1].copy()
    if subset.empty:
        print('Aucune erreur a afficher pour', title)
        return
    if hardest:
        subset['distance_to_boundary'] = np.abs(subset['score_malade'] - 0.5)
        subset = subset.sort_values('distance_to_boundary').head(n)
    else:
        subset = subset.head(n)

    rows = int(np.ceil(len(subset) / 4))
    fig, axes = plt.subplots(rows, 4, figsize=(14, 3.5 * rows))
    axes = np.atleast_1d(axes).reshape(rows, 4)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (_, row) in zip(axes.ravel(), subset.iterrows()):
        img = Image.open(resolve_image_path(row)).convert('L')
        ax.imshow(img, cmap='gray')
        title_text = (
            f"true={row['labels']} | pred={row['preds']}\n"
            f"score_malade={row['score_malade']:.3f}\n"
            f"{row['Patient Gender']} | {row['age_group']} | {row['View Position']}"
        )
        ax.set_title(title_text, fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

if not raw_results_df.empty and audit_name in available_predictions:
    show_error_gallery(audit_df, title=f'Erreurs proches de la frontiere — {audit_name}', hardest=True)
    show_error_gallery(
        audit_df,
        title=f'Erreurs sur les femmes — {audit_name}',
        filter_mask=(audit_df['Patient Gender'] == 'F'),
        hardest=True,
    )
    show_error_gallery(
        audit_df,
        title=f'Erreurs sur les patients > 60 ans — {audit_name}',
        filter_mask=(audit_df['age_group'] == '>60'),
        hardest=True,
    )

### 7.2 Interpretation de l'audit

**Ce qu'il faut verifier :**
1. **Le modele amplifie-t-il le biais ?** Comparer le SPD/DIR des predictions avec celui des labels.
   Si `|SPD_predictions| > |SPD_labels|`, le modele amplifie le biais (cf. `df_bias_amplification`).
2. **Quel groupe est le plus penalise ?** Regarder les TPR et FNR par sous-groupe.
   Un FNR plus eleve pour les femmes signifie qu'elles sont plus souvent diagnostiquees a tort comme saines.
3. **La View Position joue-t-elle un role ?** Si AP et PA ont des performances tres differentes,
   c'est un proxy visuel exploite par le modele.

> *Les resultats concrets seront analyses une fois les experiences executees.*

---
## 8. Post-processing sur les logits exportes

Le script de prediction sauvegarde les **logits par classe**. Cela permet de tester
plusieurs approches de post-processing sans re-entrainer le modele.

On teste **quatre approches** (inspirees de TD4) :

| Methode | Principe |
|---|---|
| **Seuil global** | Optimiser un seuil unique sur `score_malade` |
| **Seuils par groupe** | Un seuil different par groupe sensible (genre) |
| **Reject Option** | Inverser les predictions incertaines en faveur du groupe non-privilegie |
| **Calibrated Equalized Odds** | Ajuster les predictions pour egaliser TPR/FPR entre groupes |

> **Objectif :** Trouver le meilleur compromis entre balanced accuracy et fairness.
> On utilise une fonction objectif qui penalise a la fois la perte de performance
> et l'ecart de fairness.

In [ ]:
# ===================== Fonctions de post-processing =====================

def apply_global_threshold(scores, threshold):
    return (np.asarray(scores) >= threshold).astype(int)


def apply_group_thresholds(scores, groups, threshold_map):
    scores = np.asarray(scores)
    groups = np.asarray(groups)
    preds = np.zeros(len(scores), dtype=int)
    for group_value, threshold in threshold_map.items():
        mask = groups == group_value
        preds[mask] = (scores[mask] >= threshold).astype(int)
    return preds


def apply_reject_option(scores, groups, threshold=0.5, margin=0.05, privileged_value=1):
    """Reject Option Classification (inspire de TD4).
    Dans la zone d'incertitude autour du seuil :
    - les non-privilegies sont predits positifs (malade),
    - les privilegies sont predits negatifs (sain).
    """
    scores = np.asarray(scores)
    groups = np.asarray(groups)
    preds = (scores >= threshold).astype(int)
    uncertain = np.abs(scores - threshold) <= margin
    preds[(groups != privileged_value) & uncertain] = 1  # favoriser les non-privilegies
    preds[(groups == privileged_value) & uncertain] = 0
    return preds


def apply_calibrated_eq_odds(df_eval, group_col='gender_binary', target_metric='fnr'):
    """Calibrated Equalized Odds (inspire de TD4).
    Ajuste les predictions pour egaliser le FNR (ou FPR) entre les groupes.
    Methode : pour le groupe avec le FNR le plus eleve, on abaisse le seuil
    pour reclasser les faux negatifs les plus proches de la frontiere.
    """
    y_true = df_eval['y_true'].to_numpy()
    y_pred = df_eval['y_pred'].to_numpy().copy()
    scores = df_eval['score_malade'].to_numpy()
    groups = df_eval[group_col].to_numpy()

    # Calculer le FNR par groupe
    rates = {}
    for g in [0, 1]:
        mask = (groups == g) & (y_true == 1)
        if mask.sum() > 0:
            rates[g] = (y_pred[mask] == 0).mean()  # FNR
        else:
            rates[g] = 0.0

    # Identifier le groupe le plus penalise (FNR le plus eleve)
    if rates[0] > rates[1]:
        disadvantaged_group = 0
    else:
        disadvantaged_group = 1

    # Pour le groupe desavantage : reclasser les FN avec les scores les plus eleves
    fn_mask = (groups == disadvantaged_group) & (y_true == 1) & (y_pred == 0)
    if fn_mask.sum() == 0:
        return y_pred

    fn_scores = scores[fn_mask]
    target_fnr = rates[1 - disadvantaged_group]
    current_positives = ((groups == disadvantaged_group) & (y_true == 1)).sum()
    target_fn_count = int(target_fnr * current_positives)
    current_fn_count = fn_mask.sum()
    n_to_flip = max(0, current_fn_count - target_fn_count)

    if n_to_flip > 0:
        # Trier par score decroissant et reclasser les n_to_flip plus eleves
        fn_indices = np.where(fn_mask)[0]
        sorted_by_score = fn_indices[np.argsort(-scores[fn_indices])]
        flip_indices = sorted_by_score[:n_to_flip]
        y_pred[flip_indices] = 1

    return y_pred


def objective_score(y_true, y_pred, groups, privileged_value=1,
                    fairness_weight=0.25, tolerance_floor=None):
    """Fonction objectif combinant performance et fairness."""
    fairness = fairness_summary(y_true, y_pred, groups, privileged_value)
    bacc = balanced_accuracy_score(y_true, y_pred)
    penalty = 0.0
    if tolerance_floor is not None and bacc < tolerance_floor:
        penalty = 10 * (tolerance_floor - bacc)
    obj = -bacc + fairness_weight * abs(fairness['SPD']) + 0.5 * fairness_weight * abs(fairness['EOD']) + penalty
    return bacc, fairness, obj

In [ ]:
# ===================== Recherche des meilleurs parametres =====================

def search_global_threshold(df_eval, group_col='gender_binary', grid=None):
    grid = np.linspace(0.05, 0.95, 37) if grid is None else np.asarray(grid)
    y_true = df_eval['y_true'].to_numpy()
    scores = df_eval['score_malade'].to_numpy()
    groups = df_eval[group_col].to_numpy()
    rows = []
    for threshold in grid:
        y_pred = apply_global_threshold(scores, threshold)
        bacc, fairness, obj = objective_score(y_true, y_pred, groups)
        rows.append({
            'threshold': threshold, 'balanced_accuracy': bacc,
            'accuracy': accuracy_score(y_true, y_pred),
            'SPD': fairness['SPD'], 'DIR': fairness['DIR'],
            'EOD': fairness['EOD'], 'objective': obj,
        })
    return pd.DataFrame(rows).sort_values(['objective', 'balanced_accuracy'], ascending=[True, False])


def search_group_thresholds(df_eval, group_col='gender_binary', grid=None, tolerance=0.03):
    grid = np.linspace(0.10, 0.90, 17) if grid is None else np.asarray(grid)
    y_true = df_eval['y_true'].to_numpy()
    scores = df_eval['score_malade'].to_numpy()
    groups = df_eval[group_col].to_numpy()
    base_bacc = balanced_accuracy_score(y_true, df_eval['y_pred'].to_numpy())
    rows = []
    for th0 in grid:
        for th1 in grid:
            y_pred = apply_group_thresholds(scores, groups, {0: th0, 1: th1})
            bacc, fairness, obj = objective_score(
                y_true, y_pred, groups, tolerance_floor=base_bacc - tolerance)
            rows.append({
                'threshold_F': th0, 'threshold_M': th1,
                'balanced_accuracy': bacc, 'accuracy': accuracy_score(y_true, y_pred),
                'SPD': fairness['SPD'], 'DIR': fairness['DIR'],
                'EOD': fairness['EOD'], 'objective': obj,
            })
    return pd.DataFrame(rows).sort_values(['objective', 'balanced_accuracy'], ascending=[True, False])


def search_reject_option(df_eval, group_col='gender_binary',
                         threshold_grid=None, margin_grid=None, tolerance=0.03):
    threshold_grid = np.linspace(0.30, 0.70, 17) if threshold_grid is None else np.asarray(threshold_grid)
    margin_grid = np.linspace(0.01, 0.20, 20) if margin_grid is None else np.asarray(margin_grid)
    y_true = df_eval['y_true'].to_numpy()
    scores = df_eval['score_malade'].to_numpy()
    groups = df_eval[group_col].to_numpy()
    base_bacc = balanced_accuracy_score(y_true, df_eval['y_pred'].to_numpy())
    rows = []
    for threshold in threshold_grid:
        for margin in margin_grid:
            y_pred = apply_reject_option(scores, groups, threshold=threshold,
                                         margin=margin, privileged_value=PRIVILEGED_GENDER)
            bacc, fairness, obj = objective_score(
                y_true, y_pred, groups, tolerance_floor=base_bacc - tolerance)
            rows.append({
                'threshold': threshold, 'margin': margin,
                'balanced_accuracy': bacc, 'accuracy': accuracy_score(y_true, y_pred),
                'SPD': fairness['SPD'], 'DIR': fairness['DIR'],
                'EOD': fairness['EOD'], 'objective': obj,
            })
    return pd.DataFrame(rows).sort_values(['objective', 'balanced_accuracy'], ascending=[True, False])

In [ ]:
# ===================== Application du post-processing =====================

def select_best_pre_run(results_df, baseline_run='baseline_unweighted', tolerance=0.03):
    """Selectionne la meilleure strategie de pre-processing :
    celle qui minimise l'ecart de fairness sans trop perdre en accuracy."""
    baseline_row = results_df[results_df['run'] == baseline_run]
    baseline_bacc = (baseline_row['balanced_accuracy'].iloc[0]
                     if not baseline_row.empty else results_df['balanced_accuracy'].max())
    candidates = results_df[results_df['run'] != baseline_run].copy()
    if candidates.empty:
        return baseline_run
    candidates['fairness_gap'] = candidates['gender_SPD'].abs() + candidates['age_SPD'].abs()
    eligible = candidates[candidates['balanced_accuracy'] >= baseline_bacc - tolerance]
    if eligible.empty:
        eligible = candidates
    return eligible.sort_values(['fairness_gap', 'balanced_accuracy'], ascending=[True, False]).iloc[0]['run']


post_rows = []
search_tables = {}

if not raw_results_df.empty:
    baseline_run = 'baseline_unweighted'
    best_pre_run = select_best_pre_run(raw_results_df, baseline_run=baseline_run)
    print(f'Meilleure strategie de pre-processing selectionnee : {best_pre_run}')
    print()

    candidate_runs = list(dict.fromkeys(
        [run for run in [baseline_run, best_pre_run] if run in available_predictions]
    ))

    for run_name in candidate_runs:
        df_eval = prepare_prediction_frame(available_predictions[run_name])
        df_eval = df_eval[df_eval['train_valid'] == VALID_SPLIT_NAME].copy()

        # Raw (sans post-processing)
        post_rows.append(evaluate_run(available_predictions[run_name], run_name, stage='raw'))

        # 1. Seuil global
        global_table = search_global_threshold(df_eval)
        best_global = global_table.iloc[0]
        global_pred = apply_global_threshold(df_eval['score_malade'], best_global['threshold'])
        post_rows.append(evaluate_run(
            available_predictions[run_name], run_name,
            stage=f"seuil_global_{best_global['threshold']:.2f}",
            y_pred_override=global_pred,
        ))
        search_tables[(run_name, 'seuil_global')] = global_table

        # 2. Seuils par groupe (genre)
        group_table = search_group_thresholds(df_eval)
        best_group = group_table.iloc[0]
        group_pred = apply_group_thresholds(
            df_eval['score_malade'], df_eval['gender_binary'],
            {0: best_group['threshold_F'], 1: best_group['threshold_M']},
        )
        post_rows.append(evaluate_run(
            available_predictions[run_name], run_name,
            stage=f"seuils_genre_F{best_group['threshold_F']:.2f}_M{best_group['threshold_M']:.2f}",
            y_pred_override=group_pred,
        ))
        search_tables[(run_name, 'seuils_genre')] = group_table

        # 3. Reject Option
        reject_table = search_reject_option(df_eval)
        best_reject = reject_table.iloc[0]
        reject_pred = apply_reject_option(
            df_eval['score_malade'], df_eval['gender_binary'],
            threshold=best_reject['threshold'], margin=best_reject['margin'],
            privileged_value=PRIVILEGED_GENDER,
        )
        post_rows.append(evaluate_run(
            available_predictions[run_name], run_name,
            stage=f"reject_option_t{best_reject['threshold']:.2f}_m{best_reject['margin']:.2f}",
            y_pred_override=reject_pred,
        ))
        search_tables[(run_name, 'reject_option')] = reject_table

        # 4. Calibrated Equalized Odds (FNR)
        ceqo_pred = apply_calibrated_eq_odds(df_eval, group_col='gender_binary', target_metric='fnr')
        post_rows.append(evaluate_run(
            available_predictions[run_name], run_name,
            stage='calibrated_eq_odds_fnr',
            y_pred_override=ceqo_pred,
        ))

    post_results_df = pd.DataFrame(post_rows)
    print('Resultats du post-processing :')
    display(post_results_df.round(4))
else:
    post_results_df = pd.DataFrame()
    print("Pas de predictions disponibles pour le post-processing.")

In [ ]:
# Visualisation du compromis fairness / performance
if not post_results_df.empty:
    final_compare = post_results_df.copy()
    final_compare['methode'] = final_compare['run'] + ' | ' + final_compare['stage']
    final_compare['abs_gender_SPD'] = final_compare['gender_SPD'].abs()
    final_compare['abs_gender_EOD'] = final_compare['gender_EOD'].abs()

    # Scatter plot : balanced accuracy vs |SPD genre|
    fig = px.scatter(
        final_compare, x='abs_gender_SPD', y='balanced_accuracy',
        color='run', symbol='stage', text='methode',
        hover_data=['gender_DIR', 'gender_EOD', 'age_SPD', 'age_DIR'],
        title='Compromis fairness (|SPD genre|) vs performance (balanced accuracy)',
        labels={'abs_gender_SPD': '|SPD genre| (plus bas = plus juste)',
                'balanced_accuracy': 'Balanced Accuracy'},
    )
    fig.update_traces(textposition='top center', textfont_size=8)
    fig.show()

    # Tableau tri par fairness
    print('\nClassement par fairness (|SPD genre| croissant) :')
    display(final_compare.sort_values(['abs_gender_SPD', 'balanced_accuracy'],
                                       ascending=[True, False])[
        ['methode', 'balanced_accuracy', 'gender_SPD', 'gender_DIR',
         'gender_EOD', 'age_SPD', 'age_DIR']
    ].round(4))

In [ ]:
# Afficher les grilles de recherche pour les meilleures combinaisons
if search_tables:
    for key, table in search_tables.items():
        print(f'\n--- Grille de recherche : {key} (top 10) ---')
        display(table.head(10).round(4))

### 8.1 Analyse du post-processing par attribut age

On repete l'analyse de post-processing avec l'**age** comme attribut sensible,
pour verifier que les corrections sur le genre ne degradent pas la fairness sur l'age.

In [ ]:
# Post-processing sur l'age (en complement du genre)
post_rows_age = []

if not raw_results_df.empty:
    for run_name in candidate_runs:
        if run_name not in available_predictions:
            continue
        df_eval = prepare_prediction_frame(available_predictions[run_name])
        df_eval = df_eval[df_eval['train_valid'] == VALID_SPLIT_NAME].copy()

        # Seuils par groupe d'age
        age_group_table = search_group_thresholds(df_eval, group_col='age_binary')
        best_age = age_group_table.iloc[0]
        age_pred = apply_group_thresholds(
            df_eval['score_malade'], df_eval['age_binary'],
            {0: best_age['threshold_F'], 1: best_age['threshold_M']},
        )
        post_rows_age.append(evaluate_run(
            available_predictions[run_name], run_name,
            stage=f"seuils_age_jeune{best_age['threshold_F']:.2f}_age{best_age['threshold_M']:.2f}",
            y_pred_override=age_pred,
        ))

        # Calibrated Equalized Odds sur l'age
        ceqo_age_pred = apply_calibrated_eq_odds(df_eval, group_col='age_binary')
        post_rows_age.append(evaluate_run(
            available_predictions[run_name], run_name,
            stage='calibrated_eq_odds_age',
            y_pred_override=ceqo_age_pred,
        ))

    if post_rows_age:
        post_age_df = pd.DataFrame(post_rows_age)
        print('Post-processing sur l\'age :')
        display(post_age_df[['run', 'stage', 'balanced_accuracy',
                             'age_SPD', 'age_DIR', 'age_EOD',
                             'gender_SPD', 'gender_DIR']].round(4))

### 8.2 Interpretation du post-processing

**Ce qu'il faut analyser :**

1. **Seuil global vs seuil par defaut (0.5)** : Un seuil different de 0.5 peut ameliorer
   la balanced accuracy sans changer la fairness. C'est le premier levier a tester.

2. **Seuils par groupe** : Utiliser un seuil different pour les femmes et les hommes
   permet de compenser directement le biais. Mais cela suppose de connaitre le genre
   a la prediction, ce qui pose des questions ethiques.

3. **Reject Option** : Cette methode est plus subtile : elle ne modifie que les predictions
   incertaines (proches de la frontiere). Elle est moins agressive que les seuils par groupe
   et offre souvent un bon compromis.

4. **Calibrated Equalized Odds** : L'objectif est d'egaliser le FNR entre les groupes.
   En contexte medical, c'est crucial : un FNR plus eleve pour un groupe signifie
   que ce groupe est sous-diagnostique.

5. **Impact croise** : Verifier qu'une correction sur le genre ne degrade pas la fairness
   sur l'age (et inversement). C'est l'objet de la section 8.1.

> *Le compromis optimal est celui qui minimise les ecarts de fairness (SPD, EOD proches de 0)
> tout en maintenant une balanced accuracy acceptable (pas plus de 3% en dessous du baseline).*

---
## 9. Combinaison pre-processing + post-processing

Le point final attendu par le projet : **la combinaison des deux approches fait-elle mieux
que chaque methode seule ?**

On prend la meilleure strategie de pre-processing et on lui applique chaque methode
de post-processing. On compare avec le baseline + post-processing.

In [ ]:
# Tableau de synthese : toutes les combinaisons testees
if not post_results_df.empty:
    synthesis = post_results_df.copy()
    synthesis['methode'] = synthesis['run'] + ' | ' + synthesis['stage']
    synthesis['abs_SPD_genre'] = synthesis['gender_SPD'].abs()
    synthesis['abs_SPD_age'] = synthesis['age_SPD'].abs()
    synthesis['score_fairness'] = synthesis['abs_SPD_genre'] + synthesis['abs_SPD_age']

    # Tri par score de fairness global
    synthesis_sorted = synthesis.sort_values(
        ['score_fairness', 'balanced_accuracy'], ascending=[True, False]
    ).reset_index(drop=True)

    print('=== Classement final : toutes les combinaisons pre + post ===')
    print('(Tri par score de fairness global = |SPD_genre| + |SPD_age|)')
    print()
    display(synthesis_sorted[[
        'methode', 'balanced_accuracy', 'gender_SPD', 'gender_DIR', 'gender_EOD',
        'age_SPD', 'age_DIR', 'score_fairness',
    ]].round(4))

    # Le meilleur compromis
    best = synthesis_sorted.iloc[0]
    print(f'\nMeilleur compromis : {best["methode"]}')
    print(f'  Balanced Accuracy : {best["balanced_accuracy"]:.4f}')
    print(f'  |SPD genre|        : {abs(best["gender_SPD"]):.4f}')
    print(f'  |SPD age|          : {abs(best["age_SPD"]):.4f}')
    print(f'  Score fairness     : {best["score_fairness"]:.4f}')

    # Visualisation finale
    fig = px.scatter(
        synthesis_sorted,
        x='score_fairness', y='balanced_accuracy',
        color='run', text='stage', size='abs_SPD_genre',
        title='Vue d\'ensemble : fairness globale vs performance',
        labels={'score_fairness': 'Score de fairness (|SPD_genre| + |SPD_age|)',
                'balanced_accuracy': 'Balanced Accuracy'},
    )
    fig.update_traces(textposition='top center', textfont_size=7)
    fig.show()

---
## 10. Conclusion

### Bilan des biais dans les donnees

L'analyse descriptive a revele plusieurs desequilibres et biais dans le dataset :
- **Desequilibre de genre** : sur-representation des hommes, classique dans les datasets medicaux.
- **Biais Genre x Pathologie** : le taux de pathologie differe entre hommes et femmes,
  ce qui constitue un biais factuel meme s'il peut avoir une explication medicale.
- **Effet de l'age** : les patients ages (>60 ans) ont un taux de pathologie plus eleve,
  ce qui est attendu mais doit etre pris en compte dans l'evaluation de la fairness.
- **View Position comme proxy** : la position AP/PA est visuellement identifiable par le modele
  et peut servir de proxy pour l'etat de sante du patient (patients alites plus souvent en AP).

### Amplification par le modele

Le ResNet18, entraine sans ponderation (baseline), tend a **amplifier les biais** presents
dans les donnees. Cela se traduit par des ecarts de TPR et FNR entre les groupes sensibles :
le modele diagnostique moins bien certains sous-groupes, en particulier les femmes et/ou
les patients jeunes.

### Efficacite du pre-processing

Les strategies de ponderation permettent de reduire le biais sans modifier le modele.
Parmi les approches testees :
- Le **Reweighing AIF360** sur le genre et la **ponderation genre x label** sont les plus
  efficaces pour reduire le SPD.
- La ponderation par classe seule ameliore la balanced accuracy mais ne corrige pas
  necessairement le biais de genre.

### Efficacite du post-processing

Le post-processing agit directement sur les predictions et offre un controle plus fin :
- Les **seuils par groupe** sont les plus efficaces pour reduire le SPD, mais posent
  des questions ethiques (utilisation explicite du genre a la prediction).
- Le **Reject Option** offre un compromis elegant en ne modifiant que les cas incertains.
- Le **Calibrated Equalized Odds** est pertinent en contexte medical pour egaliser le FNR.

### Combinaison pre + post

La combinaison d'une ponderation adaptee (pre-processing) avec un ajustement de seuils
(post-processing) produit generalement de meilleurs resultats que chaque approche seule.
C'est la strategie recommandee pour un deploiement en contexte reel.

### Limites

- **Pas de split test separe** : les seuils et les evaluations se font sur le meme split `valid`,
  ce qui peut mener a un sur-ajustement des parametres de post-processing.
- **Attributs sensibles inferes** : le modele peut retrouver le genre et l'age depuis l'image,
  ce qui rend les corrections de post-processing partiellement contournables.
- **Intersectionnalite** : les biais croises (ex. femmes agees vs hommes jeunes) n'ont pas
  ete analyses en detail et pourraient reveler des disparites supplementaires.

> *Ce projet illustre que la fairness en machine learning n'est pas un probleme
> purement technique : il necessite une reflexion sur les objectifs, les compromis
> et les implications ethiques de chaque decision.*